# AquaWatch: train the swimmer detector on Colab

This notebook trains AquaWatch's **swimmer posture model** (`best.pt`, YOLO11n with 5 body-state classes) on public
pool datasets from Roboflow Universe (CC BY 4.0), on a free Colab T4 GPU in about 30-40 minutes.

**How to use it**
1. `Runtime > Change runtime type > T4 GPU`, then save.
2. Run the cells from top to bottom (Shift+Enter).
3. Section 4 needs your Roboflow API key in Colab **Secrets** (never paste it into a cell).

Running AquaWatch itself (the dashboard and the command line) is described in the repository's README.

## 1. Check the GPU
You should see a Tesla T4 and `CUDA available: True`. Without a GPU everything still works, only slower
(then use `--device cpu` below).

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code
Use **Option A or Option B, not both**, then run the `%cd` cell.

* **Option A (git):** clone the repository (the next cell).
* **Option B (upload):** on your laptop zip the project folder **without** `.venv`, `outputs` and `models`, then upload it.

In [ ]:
# Option A: clone from git
!git clone https://github.com/Lokeshraj2006/HexTech-AquaWatch.git /content/aquawatch

In [ ]:
# Option B: upload the zipped project (a dialog opens)
import glob, os, shutil, zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
shutil.rmtree('/content/_unzip', ignore_errors=True)
with zipfile.ZipFile(zip_name) as zf:                 # extract by hand: Windows zips may use '\\' in paths
    for info in zf.infolist():
        name = info.filename.replace('\\', '/')
        target = os.path.join('/content/_unzip', name)
        if name.endswith('/'):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with zf.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)
found = glob.glob('/content/_unzip/**/app.py', recursive=True)      # app.py sits in the project folder
assert found, 'app.py not found inside the zip - did you zip the project folder?'
shutil.rmtree('/content/aquawatch', ignore_errors=True)
shutil.move(os.path.dirname(found[0]), '/content/aquawatch')
print('code is in /content/aquawatch')

In [ ]:
%cd /content/aquawatch
!ls

## 3. Install the libraries
Colab already has PyTorch. This adds Ultralytics (YOLO11 + ByteTrack), the tracker's `lap` solver, and the rest.
If pip asks you to restart the runtime, do `Runtime > Restart session` and continue from the next cell.

In [ ]:
!pip install -q -r requirements.txt
import ultralytics
ultralytics.checks()

## 4. Train the swimmer detector (about 30-40 min on a T4)

Downloads the public pool datasets (Roboflow Universe, CC BY 4.0), merges them into our 5 body-state classes
(duplicates removed, split by video so the test score is honest), fine-tunes YOLO11n, evaluates it on the
test split and gives you `best.pt` (see training/README.md for how the dashboard uses it).

**Your Roboflow key stays private:** click the **key icon (Secrets)** in Colab's left sidebar, add a secret named
`ROBOFLOW_API_KEY` with your *Private API Key*, and allow this notebook to access it. Never paste the key into a cell.

In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')   # read from Colab Secrets, never printed
!python training/get_datasets.py
!python training/build_dataset.py

In [ ]:
# Full training on the T4 (lower --epochs if you are short of time; 30 epochs ~ 20 min)
!python training/train_swimmer.py --data datasets/swimmers/data.yaml --epochs 60 --device 0

In [ ]:
# Look at the training curves and the confusion matrix, then download the model
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png']:
    p = f'training/runs/swimmer/{f}'
    if os.path.exists(p):
        display(Image(p, width=900))
from google.colab import files
files.download('training/runs/swimmer/weights/best.pt')
files.download('training/runs/swimmer/eval.json')